# Notebook 00: Environment Setup and Verification

This notebook verifies your development environment for the CCA Customer Support Resolution Agent project.

Run each cell in order. Green checkmarks mean you're good; red X means something needs fixing.

## Notebook Template Convention

Every notebook in this series follows the same structure: **Setup > Anti-Pattern > Correct Pattern > Compare**.

We show the anti-pattern first so you can *feel the pain* before seeing the fix. This mirrors the CCA exam format where wrong answers look plausible.

<div style="border-left: 4px solid #dc3545; padding: 12px 16px; background: #fff5f5; margin: 8px 0;">
<strong>What's wrong:</strong> This red box marks anti-pattern sections in every notebook. The code below demonstrates a common CCA mistake — it looks reasonable but fails under specific conditions.
</div>

<div style="border-left: 4px solid #28a745; padding: 12px 16px; background: #f0fff4; margin: 8px 0;">
<strong>Why this works:</strong> This green box marks correct-pattern sections. The code below follows CCA best practices with programmatic enforcement, not just prompt instructions.
</div>

---
## Environment Checks

In [ ]:
import sys
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

# Load .env file (find_dotenv walks up directories)
load_dotenv(find_dotenv(), override=False)

# Status indicators
OK = "\033[92m[OK]\033[0m"
FAIL = "\033[91m[FAIL]\033[0m"

checks_passed = 0
checks_total = 4

In [ ]:
# Check 1: Python version
major, minor, micro = sys.version_info[:3]
version_str = f"{major}.{minor}.{micro}"
min_version = (3, 13)
if (major, minor) < min_version:
    print(f"{FAIL} Python {version_str} — need 3.13+")
    print("    Fix: Install Python 3.13 via pyenv or python.org")
    print("    Then: poetry env use python3.13 && poetry install")
else:
    print(f"{OK} Python {version_str}")
    checks_passed += 1

In [ ]:
# Check 2: Anthropic SDK installed and importable
try:
    import anthropic

    version = anthropic.__version__
    print(f"{OK} anthropic SDK {version}")
    checks_passed += 1
except ImportError:
    print(f"{FAIL} anthropic SDK not installed")
    print("    Fix: poetry install")

In [ ]:
# Check 3: ANTHROPIC_API_KEY set and valid
import os

api_key = os.environ.get("ANTHROPIC_API_KEY", "")
if not api_key or api_key == "your-api-key-here":
    print(f"{FAIL} ANTHROPIC_API_KEY not set or still placeholder")
    print("    Fix: Copy .env.example to .env and add your key:")
    print("    cp .env.example .env")
    print("    Then edit .env with your actual API key from console.anthropic.com")
else:
    try:
        client = anthropic.Anthropic()
        response = client.messages.create(
            model="claude-sonnet-4-6-20260217",
            max_tokens=10,
            messages=[{"role": "user", "content": "Say OK"}],
        )
        print(f"{OK} API key valid — Claude responded")
        checks_passed += 1

        # Show token usage as a preview of the print_usage helper
        sys.path.insert(0, str(Path(".").resolve()))
        from helpers import print_usage

        print_usage(response)
    except anthropic.AuthenticationError:
        print(f"{FAIL} API key is invalid")
        print("    Fix: Check your key at console.anthropic.com/settings/keys")
    except Exception as e:
        print(f"{FAIL} API call failed: {e}")
        print("    This might be a network issue. Check your internet connection.")

In [ ]:
# Check 4: Package and all sub-packages importable
sub_packages = ["agent", "anti_patterns", "data", "models", "services", "tools"]
all_ok = True

try:
    import customer_service

    print(f"{OK} customer_service {customer_service.__version__}")
except ImportError:
    print(f"{FAIL} customer_service not importable")
    print("    Fix: poetry install")
    all_ok = False

if all_ok:
    for pkg in sub_packages:
        try:
            __import__(f"customer_service.{pkg}")
            print(f"  {OK} customer_service.{pkg}")
        except ImportError:
            print(f"  {FAIL} customer_service.{pkg}")
            all_ok = False

if all_ok:
    checks_passed += 1
else:
    print("    Fix: poetry install (from project root)")

---
## Summary

In [ ]:
# Final summary
sep = "=" * 50
print(f"\n{sep}")
print(f"  Environment Check: {checks_passed}/{checks_total} passed")
print(f"{sep}")

if checks_passed == checks_total:
    print("\n\033[92mAll checks passed! You're ready to start the course.\033[0m")
    print("Next: Open notebook 01 (Escalation Pattern)")
else:
    failed = checks_total - checks_passed
    msg = f"\033[91m{failed} check(s) failed. Fix the issues above and re-run.\033[0m"
    print(f"\n{msg}")

---

<div style="border-left: 4px solid #007bff; padding: 12px 16px; background: #f0f7ff; margin: 8px 0;">
<strong>CCA Exam Tip:</strong> The CCA exam tests whether you can identify correct architectural patterns vs common mistakes. Each notebook in this series shows you both — the anti-pattern (what NOT to do) and the correct approach (what TO do). Pay attention to <em>why</em> each pattern fails or succeeds.
</div>